<a href="https://colab.research.google.com/github/buixuanhieu1999/finance_table_exxtract/blob/main/0000.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Financial Statement Extraction Notebook

Pure Python pipeline:
1. Render only the first few PDF pages for TOC detection.
2. Use an Ollama Cloud vision model to detect the table of contents.
3. Convert TOC document pages into real PDF/system pages.
4. Create a small TOC-selected subset PDF and send only that to MinerU.
5. Normalize and semantically chunk MinerU markdown.
6. Send the chunks to another Ollama Cloud model for financial statement extraction.

Install runtime dependencies if needed:

```bash
pip install pymupdf requests
```

Services expected:
- Ollama Cloud: `https://ollama.com` with `OLLAMA_API_KEY`
- MinerU API: local or tunneled URL ending in `/file_parse`


## Optional Colab Bootstrap: MinerU In Colab

Skip this section when MinerU is running on your own local machine.

Colab's `localhost` is the Colab VM, not your laptop. If MinerU is local on your
computer and the notebook runs in Colab, expose MinerU with a tunnel and set
`LOCAL_MINERU_URL` in the config section below.

Only set `START_MINERU_IN_COLAB = True` if you actually want Colab to install
and start MinerU inside the Colab VM.

In [ ]:
!pip install pymupdf requests

In [ ]:
# Optional Colab MinerU setup cell. Safe to run outside Colab; it will skip itself.
import os as _setup_os
import subprocess as _setup_subprocess
import sys as _setup_sys
import time as _setup_time

_IN_COLAB = "google.colab" in _setup_sys.modules
START_MINERU_IN_COLAB = False

if not _IN_COLAB:
    print("Not running in Colab; skipping Colab package/bootstrap setup.")
elif not START_MINERU_IN_COLAB:
    print("START_MINERU_IN_COLAB is False; using external/local MinerU instead.")
else:
    _setup_subprocess.run(
        [_setup_sys.executable, "-m", "pip", "install", "-q", "-U", "pip"],
        check=True,
    )
    _setup_subprocess.run(
        [_setup_sys.executable, "-m", "pip", "install", "-q", "pymupdf", "requests"],
        check=True,
    )
    _setup_subprocess.run(
        [_setup_sys.executable, "-m", "pip", "install", "-q", "-U", "mineru[core]"],
        check=True,
    )

    _setup_os.environ["MINERU_MODEL_SOURCE"] = "huggingface"

    import requests as _setup_requests

    mineru_log = open("/content/mineru-api.log", "w", encoding="utf-8")
    mineru_proc = _setup_subprocess.Popen(
        ["mineru-api", "--host", "0.0.0.0", "--port", "8000"],
        stdout=mineru_log,
        stderr=_setup_subprocess.STDOUT,
    )

    for _ in range(180):
        ready = False
        for _url in ("http://127.0.0.1:8000/health", "http://127.0.0.1:8000/docs"):
            try:
                _resp = _setup_requests.get(_url, timeout=2)
                if _resp.ok:
                    ready = True
                    break
            except Exception:
                pass
        if ready:
            print("MinerU API is ready at http://127.0.0.1:8000/file_parse")
            break
        _setup_time.sleep(2)
    else:
        print("MinerU did not become ready. Last log lines:")
        _setup_subprocess.run(["tail", "-80", "/content/mineru-api.log"], check=False)

START_MINERU_IN_COLAB is False; using external/local MinerU instead.


In [ ]:
import base64
import json
import mimetypes
import os
import re
import time
import unicodedata
from contextlib import ExitStack
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

import requests

try:
    import fitz  # PyMuPDF
except ImportError:
    fitz = None


@dataclass(frozen=True)
class Config:
    pdf_path: Path = Path(r"C:/path/to/financial_statement.pdf")
    output_dir: Path = Path("outputs").resolve()
    standard: str = "VAS"  # "VAS" or "IFRS"

    mineru_url: str = "http://127.0.0.1:8000/file_parse"
    ollama_host: str = "https://ollama.com"
    ollama_api_key_env: str = "OLLAMA_API_KEY"
    vision_model: str = "qwen3-vl:235b"
    extraction_model: str = "gpt-oss:120b"

    toc_scan_pages: int = 3
    fallback_page_count: int = 30
    toc_max_pages_per_section: int = 4
    toc_page_offset: int | None = None
    metadata_page_count: int = 3
    render_dpi: int = 180
    chunk_max_chars: int = 5_000
    mineru_timeout_seconds: int = 1_800
    ollama_timeout_seconds: int = 600

    mineru_lang_list: str = "ch"
    mineru_backend: str = "hybrid-auto-engine"
    mineru_input_mode: str = "pdf"  # "pdf" creates a TOC-selected subset PDF; "images" sends selected page PNGs
    mineru_image_batch_size: int = 2
    mineru_pdf_fallback: bool = True


CONFIG = Config()
CONFIG.output_dir.mkdir(parents=True, exist_ok=True)

## Local MinerU + Ollama Cloud Config

This is the main configuration for the requested setup:

- MinerU runs locally on your machine.
- Ollama model calls go to Ollama Cloud.

Start MinerU locally before running the pipeline:

```bash
mineru-api --host 127.0.0.1 --port 8000
```

Then set `OLLAMA_API_KEY` in your environment, or paste it below for quick testing.
If the notebook runs in Colab while MinerU runs on your laptop, `127.0.0.1` will not work;
expose local MinerU with a tunnel and set `LOCAL_MINERU_URL` to the tunnel `/file_parse` URL.

If Cloudflare Tunnel fails with DNS errors, use any HTTPS tunnel instead:

```bash
ngrok http 8000
```

Then set:

```python
LOCAL_MINERU_URL = " https://c311-171-251-232-71.ngrok-free.app/file_parse"
```

In [ ]:
import sys as _runtime_sys

_IN_COLAB = "google.colab" in _runtime_sys.modules
USE_LOCAL_MINERU_OLLAMA_CLOUD = True

# Local notebook/Jupyter value. If this notebook runs in Colab while MinerU runs
# on your laptop, set MINERU_URL in Colab secrets/env or paste a tunnel URL below.
LOCAL_MINERU_URL = os.environ.get("MINERU_URL") or (
    "" if _IN_COLAB else "http://127.0.0.1:8000/file_parse"
)


def normalize_mineru_url_early(url: str) -> str:
    url = (url or "").strip().rstrip("/")
    if not url:
        return ""
    if not url.endswith("/file_parse"):
        url = f"{url}/file_parse"
    return url

LOCAL_MINERU_URL = normalize_mineru_url_early(LOCAL_MINERU_URL)

# Colab + local MinerU example:
# LOCAL_MINERU_URL = "https://YOUR-MINERU-TUNNEL.trycloudflare.com/file_parse"

# Optional quick test only. Prefer setting an environment variable instead.
# os.environ["OLLAMA_API_KEY"] = "YOUR_OLLAMA_API_KEY"

if _IN_COLAB:
    from google.colab import files as _colab_files
    from google.colab import userdata as _colab_userdata

    LOCAL_MINERU_URL = normalize_mineru_url_early(_colab_userdata.get("MINERU_URL"))
    if LOCAL_MINERU_URL:
        print("Loaded MINERU_URL from Colab secrets.")
    else:
        print("MINERU_URL was not found in Colab secrets.")

    _ollama_key = _colab_userdata.get("OLLAMA_API_KEY")
    if _ollama_key:
        os.environ["OLLAMA_API_KEY"] = _ollama_key
        print("Loaded OLLAMA_API_KEY from Colab secrets.")
    else:
        print("OLLAMA_API_KEY was not found in Colab secrets.")

if USE_LOCAL_MINERU_OLLAMA_CLOUD:
    if _IN_COLAB and not LOCAL_MINERU_URL:
        print(
            "LOCAL_MINERU_URL is empty. Colab cannot reach MinerU on your laptop via localhost. "
            "Expose local MinerU with a tunnel, then set LOCAL_MINERU_URL to the tunnel /file_parse URL."
        )

    CONFIG = replace(
        CONFIG,
        output_dir=Path("/content/outputs").resolve() if _IN_COLAB else Path("outputs").resolve(),
        mineru_url=LOCAL_MINERU_URL,
        ollama_host="https://ollama.com",
        vision_model="qwen3-vl:235b",
        extraction_model="gemma3:4b-cloud",
        standard="VAS",
        render_dpi=120,
        fallback_page_count=30,
        toc_max_pages_per_section=4,
        metadata_page_count=3,
        chunk_max_chars=2500,
        mineru_timeout_seconds=1800,
        mineru_input_mode="pdf",
        mineru_image_batch_size=1,
        mineru_pdf_fallback=True,
    )

    USE_COLAB_FILE_UPLOAD = _IN_COLAB
    if _IN_COLAB and USE_COLAB_FILE_UPLOAD:
        uploaded = _colab_files.upload()
        if uploaded:
            CONFIG = replace(CONFIG, pdf_path=Path(next(iter(uploaded.keys()))).resolve())

    print(CONFIG)
else:
    print("USE_LOCAL_MINERU_OLLAMA_CLOUD is False; edit CONFIG manually.")

Loaded MINERU_URL from Colab secrets.
Loaded OLLAMA_API_KEY from Colab secrets.


Config(pdf_path=PosixPath('C:/path/to/financial_statement.pdf'), output_dir=PosixPath('/content/outputs'), standard='VAS', mineru_url='https://c311-171-251-232-71.ngrok-free.app/file_parse', ollama_host='https://ollama.com', ollama_api_key_env='OLLAMA_API_KEY', vision_model='qwen3-vl:235b', extraction_model='gemma3:4b-cloud', toc_scan_pages=3, fallback_page_count=30, toc_max_pages_per_section=4, toc_page_offset=None, metadata_page_count=3, render_dpi=120, chunk_max_chars=2500, mineru_timeout_seconds=1800, ollama_timeout_seconds=600, mineru_lang_list='ch', mineru_backend='hybrid-auto-engine', mineru_input_mode='pdf', mineru_image_batch_size=1, mineru_pdf_fallback=True)


## Colab Smoke Tests

Optional checks before running the full pipeline.

## Local Service Smoke Tests

Run these before the full pipeline. They check Ollama Cloud and your local MinerU API.

In [ ]:
def is_loopback_url(url: str) -> bool:
    host = urlparse(url).hostname
    return host in {"localhost", "127.0.0.1", "::1"}


def assert_mineru_url_is_reachable_context(config: Config = CONFIG) -> None:
    if not config.mineru_url:
        raise RuntimeError(
            "CONFIG.mineru_url is empty. If MinerU runs on your laptop and the notebook runs in Colab, "
            "expose MinerU with a tunnel and set LOCAL_MINERU_URL to the tunnel /file_parse URL."
        )

    if "google.colab" in _runtime_sys.modules and is_loopback_url(config.mineru_url):
        raise RuntimeError(
            "CONFIG.mineru_url points to localhost/127.0.0.1 while running in Colab. "
            "That points to the Colab VM, not your laptop. Start MinerU locally, expose it with a tunnel, "
            "then set LOCAL_MINERU_URL='https://YOUR-TUNNEL.../file_parse' and rerun the config cell."
        )


if not os.environ.get("OLLAMA_API_KEY"):
    print("Set OLLAMA_API_KEY before testing Ollama Cloud.")
else:
    _resp = requests.post(
        "https://ollama.com/api/generate",
        headers={"Authorization": f"Bearer {os.environ['OLLAMA_API_KEY']}"},
        json={
            "model": CONFIG.extraction_model,
            "prompt": 'Return only this JSON: {"ok":true}',
            "stream": False,
        },
        timeout=120,
    )
    print("Ollama Cloud:", _resp.status_code, _resp.text[:500])

Ollama Cloud: 200 {"model":"gemma3:4b-cloud","created_at":"2026-05-19T10:52:48.526474535Z","response":"```json\n{\"ok\":true}\n```\n","done":true,"done_reason":"stop","total_duration":268225330,"prompt_eval_count":19,"eval_count":12}



In [ ]:
_health_url = CONFIG.mineru_url.rsplit("/", 1)[0] + "/health" if CONFIG.mineru_url else "(empty mineru_url)"
try:
    assert_mineru_url_is_reachable_context(CONFIG)
    _resp = requests.get(_health_url, timeout=5)
    print("MinerU health:", _resp.status_code, _resp.text[:300])
except Exception as _exc:
    print(f"MinerU health check failed at {_health_url}: {_exc}")
    print("If MinerU is local, start it with: mineru-api --host 127.0.0.1 --port 8000")

MinerU health: 200 {"status":"healthy","version":"3.1.12","protocol_version":1,"queued_tasks":0,"processing_tasks":0,"completed_tasks":16,"failed_tasks":1,"max_concurrent_requests":3,"processing_window_size":64,"task_retention_seconds":86400,"task_cleanup_interval_seconds":300}


In [ ]:
if not Path(CONFIG.pdf_path).exists():
    print(f"PDF not found yet: {CONFIG.pdf_path}")
else:
    with Path(CONFIG.pdf_path).open("rb") as _pdf_file:
        _resp = requests.post(
            CONFIG.mineru_url,
            files={"files": (Path(CONFIG.pdf_path).name, _pdf_file, "application/pdf")},
            data={
                "return_md": "true",
                "lang_list": "ch",
                "backend": "hybrid-auto-engine",
            },
            timeout=CONFIG.mineru_timeout_seconds,
        )
    print("MinerU:", _resp.status_code, _resp.text[:500])

PDF not found yet: C:/path/to/financial_statement.pdf


## MinerU Timeout Recovery

If MinerU times out, run this cell before `run_pipeline(CONFIG)`.
It reduces image size/page count, sends images in very small batches, and lets
the pipeline fall back to the original PDF if image parsing still times out.

In [ ]:
APPLY_MINERU_TIMEOUT_RECOVERY = False

if APPLY_MINERU_TIMEOUT_RECOVERY:
    CONFIG = replace(
        CONFIG,
        render_dpi=100,
        fallback_page_count=30,
        mineru_timeout_seconds=2400,
        mineru_image_batch_size=1,
        mineru_input_mode="pdf",
        mineru_pdf_fallback=True,
    )
    print(CONFIG)
else:
    print("Timeout recovery config not applied. Set APPLY_MINERU_TIMEOUT_RECOVERY = True to use it.")

# If MinerU still times out, switch to direct PDF parsing:
#
# CONFIG = replace(
#     CONFIG,
#     mineru_input_mode="pdf",
#     mineru_timeout_seconds=2400,
# )

Timeout recovery config not applied. Set APPLY_MINERU_TIMEOUT_RECOVERY = True to use it.


## Helpers

In [ ]:
def require_pymupdf() -> None:
    if fitz is None:
        raise ImportError(
            "PyMuPDF is required for PDF-to-image rendering. Install it with: pip install pymupdf"
        )


def clean_file_stem(file_path: Path) -> str:
    return re.sub(r"[^\w.-]+", "_", file_path.stem)


def mime_type_for(file_path: Path) -> str:
    return mimetypes.guess_type(file_path.name)[0] or "application/octet-stream"


def image_to_base64(image_path: Path) -> str:
    return base64.b64encode(image_path.read_bytes()).decode("utf-8")


def strip_code_fences(text: str | None) -> str | None:
    if not text:
        return None
    text = text.strip()
    text = re.sub(r"^```(?:json|javascript|js|python)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()


def _json_start_index(text: str) -> int:
    start_candidates = [idx for idx in (text.find("{"), text.find("[")) if idx != -1]
    if not start_candidates:
        raise ValueError("No JSON object or array found")
    return min(start_candidates)


def extract_balanced_json(text: str) -> str:
    """Extract the first balanced JSON object or array from a model response."""
    text = strip_code_fences(text) or ""
    start = _json_start_index(text)
    stack: list[str] = []
    in_string = False
    escaped = False

    pairs = {"{": "}", "[": "]"}
    openers = set(pairs)
    closers = set(pairs.values())

    for idx in range(start, len(text)):
        char = text[idx]
        if in_string:
            if escaped:
                escaped = False
            elif char == "\\":
                escaped = True
            elif char == '"':
                in_string = False
            continue

        if char == '"':
            in_string = True
        elif char in openers:
            stack.append(pairs[char])
        elif char in closers:
            if not stack or char != stack[-1]:
                raise ValueError("Unbalanced JSON response")
            stack.pop()
            if not stack:
                return text[start : idx + 1]

    raise ValueError("JSON response was not closed")


def close_truncated_json(text: str) -> str:
    """Best-effort close for LLM JSON that stopped after valid content but before final ]/}."""
    text = strip_code_fences(text) or ""
    start = _json_start_index(text)
    text = text[start:].strip()

    stack: list[str] = []
    in_string = False
    escaped = False
    pairs = {"{": "}", "[": "]"}
    openers = set(pairs)
    closers = set(pairs.values())

    for char in text:
        if in_string:
            if escaped:
                escaped = False
            elif char == "\\":
                escaped = True
            elif char == '"':
                in_string = False
            continue

        if char == '"':
            in_string = True
        elif char in openers:
            stack.append(pairs[char])
        elif char in closers and stack and char == stack[-1]:
            stack.pop()

    repaired = text
    if in_string:
        if repaired.endswith("\\"):
            repaired = repaired[:-1]
        repaired += '"'

    # If the model stopped right after a key/colon/comma, remove that dangling token.
    repaired = re.sub(r',\s*$', '', repaired)
    repaired = re.sub(r'"[^"\\]*(?:\\.[^"\\]*)?"\s*:\s*$', '', repaired)
    repaired = re.sub(r',\s*([}\]])', r'\1', repaired)

    for closer in reversed(stack):
        repaired = re.sub(r',\s*$', '', repaired.rstrip())
        repaired += closer

    repaired = re.sub(r",\s*([}\]])", r"\1", repaired)
    return repaired


def light_json_repair(text: str) -> str:
    """Fix common LLM JSON noise without changing values aggressively."""
    text = strip_code_fences(text) or ""
    try:
        text = extract_balanced_json(text)
    except ValueError as exc:
        if "not closed" not in str(exc):
            raise
        text = close_truncated_json(text)
    text = re.sub(r",\s*([}\]])", r"\1", text)
    return text


def empty_none_values(value: Any) -> Any:
    if value is None:
        return ""
    if isinstance(value, list):
        return [empty_none_values(item) for item in value]
    if isinstance(value, dict):
        return {key: empty_none_values(item) for key, item in value.items()}
    return value


def parse_json_response(label: str, raw: str | None) -> Any | None:
    if not raw:
        print(f"[Parse] {label}: empty response")
        return None

    try:
        return empty_none_values(json.loads(light_json_repair(raw)))
    except Exception as exc:
        print(f"[Parse] {label}: failed: {exc}")
        print("[Parse] Response head:", (raw or "")[:600])
        print("[Parse] Response tail:", (raw or "")[-600:])
        return None


def save_json(data: Any, output_path: Path) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    return output_path


## PDF To Image

Python improvement: `PyMuPDF` renders pages directly to PNG at a controlled DPI,
with no intermediate PDF copying and no Node image package.

In [ ]:
@dataclass(frozen=True)
class PageImage:
    page_index: int
    page_number: int
    path: Path


def get_pdf_page_count(pdf_path: Path) -> int:
    require_pymupdf()
    with fitz.open(pdf_path) as doc:
        return len(doc)


def render_pdf_pages_to_images(
    pdf_path: Path,
    max_pages: int | None,
    *,
    output_dir: Path = CONFIG.output_dir,
    dpi: int = CONFIG.render_dpi,
) -> list[PageImage]:
    require_pymupdf()
    pdf_path = Path(pdf_path)
    image_dir = output_dir / f"{clean_file_stem(pdf_path)}_toc_scan_pages"
    image_dir.mkdir(parents=True, exist_ok=True)

    rendered: list[PageImage] = []
    zoom = dpi / 72
    matrix = fitz.Matrix(zoom, zoom)

    with fitz.open(pdf_path) as doc:
        total_pages = len(doc)
        page_limit = total_pages if max_pages is None else min(max_pages, total_pages)

        for page_index in range(page_limit):
            page = doc.load_page(page_index)
            pix = page.get_pixmap(matrix=matrix, alpha=False)
            image_path = image_dir / f"{clean_file_stem(pdf_path)}_page_{page_index + 1:03d}.png"
            pix.save(str(image_path))
            rendered.append(PageImage(page_index=page_index, page_number=page_index + 1, path=image_path))

    print(f"[Render] Rendered {len(rendered)}/{total_pages} page(s) for TOC scan to {image_dir}")
    return rendered


def render_pdf_page_numbers_to_images(
    pdf_path: Path,
    page_numbers: list[int],
    *,
    output_dir: Path = CONFIG.output_dir,
    dpi: int = CONFIG.render_dpi,
) -> list[PageImage]:
    require_pymupdf()
    pdf_path = Path(pdf_path)
    image_dir = output_dir / f"{clean_file_stem(pdf_path)}_selected_pages"
    image_dir.mkdir(parents=True, exist_ok=True)

    rendered: list[PageImage] = []
    zoom = dpi / 72
    matrix = fitz.Matrix(zoom, zoom)

    with fitz.open(pdf_path) as doc:
        total_pages = len(doc)
        valid_page_numbers = sorted({page for page in page_numbers if 1 <= page <= total_pages})

        for page_number in valid_page_numbers:
            page_index = page_number - 1
            page = doc.load_page(page_index)
            pix = page.get_pixmap(matrix=matrix, alpha=False)
            image_path = image_dir / f"{clean_file_stem(pdf_path)}_page_{page_number:03d}.png"
            pix.save(str(image_path))
            rendered.append(PageImage(page_index=page_index, page_number=page_number, path=image_path))

    print(f"[Render] Rendered selected page image(s): {valid_page_numbers}")
    return rendered


def create_pdf_subset(pdf_path: Path, page_numbers: list[int], output_path: Path) -> Path:
    """Create a small PDF from 1-based PDF/system page numbers."""
    require_pymupdf()
    pdf_path = Path(pdf_path)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with fitz.open(pdf_path) as src:
        total_pages = len(src)
        valid_page_numbers = sorted({page for page in page_numbers if 1 <= page <= total_pages})
        if not valid_page_numbers:
            raise ValueError("No valid pages were selected for the subset PDF")

        subset = fitz.open()
        try:
            for page_number in valid_page_numbers:
                page_index = page_number - 1
                subset.insert_pdf(src, from_page=page_index, to_page=page_index)

            if output_path.exists():
                output_path.unlink()
            subset.save(str(output_path), garbage=4, deflate=True)
        finally:
            subset.close()

    print(f"[Render] Saved TOC-selected subset PDF with pages {valid_page_numbers} to {output_path}")
    return output_path



def format_page_slug(page_numbers: list[int]) -> str:
    pages = sorted({int(page) for page in page_numbers})
    if not pages:
        return "none"
    if len(pages) <= 6:
        return "_".join(f"{page:03d}" for page in pages)
    return f"{pages[0]:03d}_{pages[-1]:03d}_{len(pages)}p"


def create_metadata_subset_pdf(
    pdf_path: Path,
    metadata_page_numbers: list[int],
    output_dir: Path,
) -> dict[str, Any] | None:
    """Create a tiny META PDF so company/date fields survive table-only MinerU runs."""
    pages = sorted({int(page) for page in metadata_page_numbers})
    if not pages:
        return None

    output_dir = Path(output_dir)
    statement_dir = output_dir / f"{clean_file_stem(pdf_path)}_statement_pdfs"
    statement_dir.mkdir(parents=True, exist_ok=True)

    out_path = statement_dir / f"{clean_file_stem(pdf_path)}_META_pages_{format_page_slug(pages)}.pdf"
    create_pdf_subset(pdf_path, pages, out_path)
    return {
        "table_key": "META",
        "title": "metadata pages",
        "pages": pages,
        "pdf_path": str(out_path),
    }



def create_statement_subset_pdfs(
    pdf_path: Path,
    selected_ranges: list[dict[str, Any]],
    selected_page_numbers: list[int],
    output_dir: Path,
) -> list[dict[str, Any]]:
    """
    Create one small PDF per financial statement when TOC ranges are available.
    This makes MinerU output separable by filename, so chunking does not depend only on OCR title text.
    """
    output_dir = Path(output_dir)
    statement_dir = output_dir / f"{clean_file_stem(pdf_path)}_statement_pdfs"
    statement_dir.mkdir(parents=True, exist_ok=True)

    statement_files: list[dict[str, Any]] = []
    if selected_ranges:
        for item in selected_ranges:
            table_key = str(item.get("table_key") or "TABLE")
            start_page = int(item["start_page"])
            end_page = int(item["end_page"])
            pages = list(range(start_page, end_page + 1))
            out_path = statement_dir / f"{clean_file_stem(pdf_path)}_{table_key}_pages_{start_page:03d}_{end_page:03d}.pdf"
            create_pdf_subset(pdf_path, pages, out_path)
            statement_files.append({**item, "pages": pages, "pdf_path": str(out_path)})
    else:
        if not selected_page_numbers:
            raise ValueError("No selected pages available for MinerU")
        start_page = min(selected_page_numbers)
        end_page = max(selected_page_numbers)
        out_path = statement_dir / f"{clean_file_stem(pdf_path)}_FALLBACK_pages_{start_page:03d}_{end_page:03d}.pdf"
        create_pdf_subset(pdf_path, selected_page_numbers, out_path)
        statement_files.append(
            {
                "table_key": "FALLBACK",
                "title": "fallback selected pages",
                "start_page": start_page,
                "end_page": end_page,
                "pages": selected_page_numbers,
                "pdf_path": str(out_path),
            }
        )

    print("[Render] Statement subset PDFs:", statement_files)
    return statement_files


## Ollama Client

In [ ]:
def ollama_generate(
    *,
    model: str,
    prompt: str,
    config: Config = CONFIG,
    system: str | None = None,
    images: list[str] | None = None,
    options: dict[str, Any] | None = None,
    retries: int = 2,
) -> str:
    payload: dict[str, Any] = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": options or {},
    }
    if system:
        payload["system"] = system
    if images:
        payload["images"] = images

    headers: dict[str, str] = {}
    api_key = os.environ.get(config.ollama_api_key_env)
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"

    url = f"{config.ollama_host.rstrip('/')}/api/generate"
    last_exc: Exception | None = None

    for attempt in range(retries + 1):
        try:
            resp = requests.post(
                url,
                json=payload,
                headers=headers,
                timeout=config.ollama_timeout_seconds,
            )
            resp.raise_for_status()
            return resp.json().get("response", "")
        except Exception as exc:
            last_exc = exc
            if attempt >= retries:
                break
            sleep_for = 1.5 * (attempt + 1)
            print(f"[Ollama] Retry {attempt + 1}/{retries} after error: {exc}")
            time.sleep(sleep_for)

    raise RuntimeError(f"Ollama call failed for {model}: {last_exc}")


def unload_ollama_model(model: str, config: Config = CONFIG) -> None:
    if config.ollama_host.rstrip("/") == "https://ollama.com":
        print(f"[Ollama] Cloud host in use; no local unload needed for {model}")
        return

    try:
        resp = requests.post(
            f"{config.ollama_host.rstrip('/')}/api/generate",
            json={"model": model, "keep_alive": 0, "stream": False},
            timeout=60,
        )
        print(f"[Ollama] Unloaded {model}, status {resp.status_code}")
    except Exception as exc:
        print(f"[Ollama] Failed to unload {model}: {exc}")

## Detect Table Of Contents

The first few rendered page images are sent to the vision model. If no TOC is found,
the pipeline falls back to the first `fallback_page_count` pages.

In [ ]:
TOC_PROMPT = """
- với mỗi trang là 1 object riêng biệt với cấu trúc như sau: { "table_of_contents": [ { "title": "", "page": "" } ] }
- chỉ cần trả về dữ liệu trong object đó và tuyệt đối không được trả thêm bất kỳ text nào khác ngoài object đó
- tuyệt đối không nhầm lẫn với THÔNG TIN NHÂN SỰ
- điểm nhận biết:
  + sẽ có 1 bảng với 2 cột, cột thứ nhất là title, cột thứ 2 là trang
  + thường có các tiêu đề như: Mục lục, Bảng cân đối kế toán, Báo cáo kết quả hoạt động kinh doanh, Báo cáo lưu chuyển tiền tệ, Thuyết minh báo cáo tài chính
- Liệt kê tên mục và số trang bắt đầu, nếu một mục có nhiều trang thì chỉ lấy trang đầu tiên và bỏ qua các trang sau.
  Ví dụ: Bảng cân đối kế toán có 2 trang là 02-03 thì trả về {"table_of_contents": [ { "title": "Bảng cân đối kế toán", "page": "2" }] }
- nếu không tìm thấy mục lục thì trả về { "table_of_contents": null }
"""

TOC_SYSTEM = (
    'Bạn là một chuyên gia trong lĩnh vực báo cáo tài chính. '
    'Hãy trích xuất thông tin mục lục từ báo cáo tài chính, chỉ quan tâm đến thông tin mục lục, không bịa ra dữ liệu.'
)


def detect_table_of_contents(page_images: list[PageImage], config: Config = CONFIG) -> list[dict[str, Any]]:
    """Detect TOC from the first rendered pages. The prompt/rules above are unchanged."""
    for page in page_images[: config.toc_scan_pages]:
        print(f"[TOC] Scanning page {page.page_number}")
        try:
            raw = ollama_generate(
                model=config.vision_model,
                system=TOC_SYSTEM,
                prompt=TOC_PROMPT,
                images=[image_to_base64(page.path)],
                options={"temperature": 0.1, "num_predict": 512, "num_ctx": 8000},
                config=config,
            )
            parsed = parse_json_response(f"TOC page {page.page_number}", raw)
            toc = parsed.get("table_of_contents") if isinstance(parsed, dict) else None
            if isinstance(toc, list) and toc:
                cleaned = []
                for item in toc:
                    if not isinstance(item, dict):
                        continue
                    title = str(item.get("title") or item.get("name") or "").strip()
                    page_value = str(item.get("page") or "").strip()
                    if title and page_value:
                        cleaned.append({"title": title, "page": page_value, "_toc_pdf_page": page.page_number})
                if cleaned:
                    print(f"[TOC] Found {len(cleaned)} item(s): {cleaned}")
                    return cleaned
        except Exception as exc:
            print(f"[TOC] Page {page.page_number} failed: {exc}")

    print("[TOC] No table of contents found")
    return []


def parse_toc_page_number(value: Any) -> int | None:
    """Parse the first page number from strings like '02', '02-03', '6 - 9', or 'Trang 2'."""
    match = re.search(r"\d+", str(value or ""))
    return int(match.group(0)) if match else None


def parse_toc_page_range(value: Any) -> tuple[int | None, int | None]:
    """Parse TOC page ranges like '6 - 9'. Returns (start, end)."""
    numbers = [int(x) for x in re.findall(r"\d+", str(value or ""))]
    if not numbers:
        return None, None
    start = numbers[0]
    end = numbers[1] if len(numbers) > 1 and numbers[1] >= start else start
    return start, end


def repair_mojibake(text: str) -> str:
    """Repair common UTF-8-as-Latin-1/CP1252 mojibake if it appears in OCR/LLM text."""
    text = text or ""
    candidates = [text]
    for encoding in ("latin1", "cp1252"):
        try:
            candidates.append(text.encode(encoding).decode("utf-8"))
        except Exception:
            pass

    bad_markers = ["\u00c3", "\u00c2", "\u00c4", "\u00c1", "\ufffd"]
    def score(candidate: str) -> int:
        bad = sum(candidate.count(marker) for marker in bad_markers)
        return sum(ord(ch) > 127 for ch in candidate) - bad * 20

    return max(candidates, key=score)


def toc_normalize(text: str) -> str:
    text = repair_mojibake(text or "")
    normalized = unicodedata.normalize("NFD", text)
    without_marks = "".join(char for char in normalized if unicodedata.category(char) != "Mn")
    return without_marks.replace("\u0111", "d").replace("\u0110", "D").lower()
def infer_toc_page_offset(table_of_contents: list[dict[str, Any]], config: Config = CONFIG) -> int:
    """
    Convert report/printed page numbers to PDF/system page numbers.

    If the TOC was detected on PDF page N and the first TOC entry starts at report page 1,
    then printed page 1 usually starts on PDF page N+1, so offset=N.
    Override with CONFIG.toc_page_offset when a file needs a known shift.
    """
    if config.toc_page_offset is not None:
        return config.toc_page_offset

    if not table_of_contents:
        return 0

    first_page = parse_toc_page_number(table_of_contents[0].get("page"))
    toc_pdf_page = table_of_contents[0].get("_toc_pdf_page")
    if first_page == 1 and isinstance(toc_pdf_page, int) and toc_pdf_page >= 1:
        return toc_pdf_page
    if first_page == 1:
        return 1
    return 0


def toc_table_patterns(standard: str) -> dict[str, list[str]]:
    if standard.upper() == "IFRS":
        return {
            "CSFP": ["statement of financial position", "financial position", "balance sheet"],
            "CSPLCI": ["profit or loss", "comprehensive income", "income statement"],
            "CSCE": ["changes in equity"],
            "CSCF": ["cash flow", "cash flows"],
        }

    return {
        "BCDKT": [
            "bang can doi ke toan",
            "bao cao tinh hinh tai chinh",
            "financial position",
            "balance sheet",
        ],
        "BCHDKD": [
            "bao cao ket qua hoat dong kinh doanh",
            "ket qua hoat dong kinh doanh",
            "bao cao ket qua hoat dong",
            "ket qua hoat dong hop nhat",
            "profit or loss",
            "income statement",
        ],
        "BCLCTT": [
            "bao cao luu chuyen tien te",
            "luu chuyen tien te",
            "cash flow",
            "cash flows",
        ],
    }


def toc_entry_table_key(title: str, standard: str) -> str | None:
    normalized_title = toc_normalize(title)
    for table_key, patterns in toc_table_patterns(standard).items():
        if any(pattern in normalized_title for pattern in patterns):
            return table_key
    return None


def select_statement_pages_from_toc(
    table_of_contents: list[dict[str, Any]],
    *,
    total_pages: int,
    config: Config = CONFIG,
) -> tuple[list[int], list[dict[str, Any]]]:
    """
    Select only financial statement pages from TOC.
    Returns 1-based PDF/system pages, not printed report page labels.
    """
    if not table_of_contents:
        fallback_pages = list(range(1, min(config.fallback_page_count, total_pages) + 1))
        print(f"[TOC] No TOC. Falling back to first {len(fallback_pages)} page(s): {fallback_pages}")
        return fallback_pages, []

    offset = infer_toc_page_offset(table_of_contents, config)
    entries: list[dict[str, Any]] = []

    for item in table_of_contents:
        title = str(item.get("title") or item.get("name") or "").strip()
        toc_page, toc_end_page = parse_toc_page_range(item.get("page"))
        if not title or toc_page is None:
            continue
        if toc_end_page is None:
            toc_end_page = toc_page
        system_page = max(1, toc_page + offset)
        system_end_page = max(system_page, toc_end_page + offset)
        if system_page <= total_pages:
            entries.append({
                **item,
                "title": title,
                "toc_page": toc_page,
                "toc_end_page": toc_end_page,
                "system_page": system_page,
                "system_end_page": min(system_end_page, total_pages),
            })

    entries.sort(key=lambda item: item["system_page"])
    selected_ranges: list[dict[str, Any]] = []

    for idx, item in enumerate(entries):
        table_key = toc_entry_table_key(item["title"], config.standard)
        if not table_key:
            continue

        start_page = item["system_page"]
        next_pages = [entry["system_page"] for entry in entries[idx + 1 :] if entry["system_page"] > start_page]
        natural_end = next_pages[0] - 1 if next_pages else item.get("system_end_page", start_page)
        explicit_end = item.get("system_end_page", start_page)
        max_end = start_page + max(1, config.toc_max_pages_per_section) - 1
        end_page = min(max(natural_end, explicit_end), max_end, total_pages)
        if end_page < start_page:
            end_page = start_page

        selected_ranges.append(
            {
                "table_key": table_key,
                "title": item["title"],
                "toc_page": item["toc_page"],
                "toc_end_page": item.get("toc_end_page", item["toc_page"]),
                "start_page": start_page,
                "end_page": end_page,
            }
        )

    if not selected_ranges:
        fallback_pages = list(range(1, min(config.fallback_page_count, total_pages) + 1))
        print(
            "[TOC] TOC found but no financial statement entries matched. "
            f"Falling back to first {len(fallback_pages)} page(s): {fallback_pages}"
        )
        return fallback_pages, []

    selected_pages = sorted(
        {
            page
            for page_range in selected_ranges
            for page in range(page_range["start_page"], page_range["end_page"] + 1)
        }
    )

    print(f"[TOC] Page offset applied: {offset}")
    print("[TOC] Selected statement ranges:", selected_ranges)
    print("[TOC] Selected PDF/system pages for MinerU:", selected_pages)
    return selected_pages, selected_ranges


def toc_entry_is_metadata(title: str, standard: str = CONFIG.standard) -> bool:
    """Detect TOC entries that usually contain company metadata, not financial tables."""
    if toc_entry_table_key(title, standard):
        return False

    normalized_title = toc_normalize(title)
    meta_terms = (
        "thong tin chung",
        "thong tin ve cong ty",
        "thong tin cong ty",
        "general information",
        "corporate information",
        "company information",
    )
    return any(term in normalized_title for term in meta_terms)


def select_metadata_pages_from_toc(
    table_of_contents: list[dict[str, Any]],
    *,
    total_pages: int,
    config: Config = CONFIG,
) -> tuple[list[int], list[dict[str, Any]]]:
    """
    Select a small set of pages for metadata extraction.

    The optimized pipeline sends only statement pages to MinerU, but company name,
    address, and report date often live on the cover or the TOC's "Thong tin chung"
    section. This keeps those pages cheap and separate as a META subset.
    """
    page_limit = max(0, int(config.metadata_page_count))
    if page_limit <= 0 or total_pages <= 0:
        return [], []

    pages: set[int] = set(range(1, min(page_limit, total_pages) + 1))
    selected_ranges: list[dict[str, Any]] = []

    if table_of_contents:
        offset = infer_toc_page_offset(table_of_contents, config)
        for item in table_of_contents:
            title = str(item.get("title") or item.get("name") or "").strip()
            if not title or not toc_entry_is_metadata(title, config.standard):
                continue

            toc_page, toc_end_page = parse_toc_page_range(item.get("page"))
            if toc_page is None:
                continue
            if toc_end_page is None:
                toc_end_page = toc_page

            start_page = max(1, toc_page + offset)
            explicit_end_page = max(start_page, toc_end_page + offset)
            max_end_page = start_page + page_limit - 1
            end_page = min(explicit_end_page, max_end_page, total_pages)
            pages.update(range(start_page, end_page + 1))
            selected_ranges.append(
                {
                    "title": title,
                    "toc_page": toc_page,
                    "toc_end_page": toc_end_page,
                    "start_page": start_page,
                    "end_page": end_page,
                }
            )

    selected_pages = sorted(page for page in pages if 1 <= page <= total_pages)
    print("[TOC] Selected metadata pages for MinerU:", selected_pages)
    if selected_ranges:
        print("[TOC] Selected metadata ranges:", selected_ranges)
    return selected_pages, selected_ranges



## MinerU OCR / Markdown

MinerU receives rendered page images by default. In Colab, images are sent in
small batches so a single heavy page range does not monopolize one long request.

In [ ]:
def normalize_mineru_url(url: str) -> str:
    url = (url or "").strip().rstrip("/")
    if not url:
        return ""
    if not url.endswith("/file_parse"):
        url = f"{url}/file_parse"
    return url


def is_loopback_url(url: str) -> bool:
    host = urlparse(url).hostname
    return host in {"localhost", "127.0.0.1", "::1"}


def assert_mineru_url_is_reachable_context(config: Config = CONFIG) -> None:
    if not config.mineru_url:
        raise RuntimeError(
            "CONFIG.mineru_url is empty. If MinerU runs on your laptop and the notebook runs in Colab, "
            "expose MinerU with a tunnel and set LOCAL_MINERU_URL to the tunnel /file_parse URL."
        )

    if "google.colab" in _runtime_sys.modules and is_loopback_url(config.mineru_url):
        raise RuntimeError(
            "CONFIG.mineru_url points to localhost/127.0.0.1 while running in Colab. "
            "That points to the Colab VM, not your laptop. Start MinerU locally, expose it with a tunnel, "
            "then set LOCAL_MINERU_URL='https://YOUR-TUNNEL.../file_parse' and rerun the config cell."
        )


def _markdown_from_mineru_payload(payload: dict[str, Any]) -> str:
    results = payload.get("results") or {}
    markdown_blocks: list[str] = []

    for name, item in results.items():
        md = (item or {}).get("md_content") or ""
        if md.strip():
            markdown_blocks.append(f"\n\n<!-- {name} -->\n\n{md.strip()}")

    if not markdown_blocks:
        raise RuntimeError(f"MinerU returned no markdown. Keys: {list(results)}")

    return "\n\n".join(markdown_blocks).strip()


def call_mineru(
    input_paths: list[Path],
    config: Config = CONFIG,
    *,
    output_name: str = "markdown-output.md",
) -> str:
    config = replace(config, mineru_url=normalize_mineru_url(config.mineru_url))
    assert_mineru_url_is_reachable_context(config)

    if not input_paths:
        raise ValueError("No files were provided to MinerU")

    input_paths = [Path(file_path) for file_path in input_paths]
    print(
        f"[MinerU] Sending {len(input_paths)} file(s) to {config.mineru_url}: "
        + ", ".join(file_path.name for file_path in input_paths[:5])
        + (" ..." if len(input_paths) > 5 else "")
    )

    with ExitStack() as stack:
        files = []
        for file_path in input_paths:
            file_path = Path(file_path)
            handle = stack.enter_context(file_path.open("rb"))
            files.append(("files", (file_path.name, handle, mime_type_for(file_path))))

        data = {
            "lang_list": config.mineru_lang_list,
            "backend": config.mineru_backend,
            "return_md": "true",
        }

        resp = requests.post(
            config.mineru_url,
            data=data,
            files=files,
            timeout=config.mineru_timeout_seconds,
        )

    if not resp.ok:
        raise RuntimeError(f"MinerU server {resp.status_code}: {resp.text[:1000]}")

    markdown = _markdown_from_mineru_payload(resp.json())
    out_path = config.output_dir / output_name
    out_path.write_text(markdown, encoding="utf-8")
    print(f"[MinerU] Markdown saved to {out_path} ({len(markdown)} chars)")
    return markdown


def call_mineru_batched(
    input_paths: list[Path],
    config: Config = CONFIG,
    *,
    batch_size: int | None = None,
) -> str:
    input_paths = [Path(file_path) for file_path in input_paths]
    batch_size = batch_size or config.mineru_image_batch_size

    if len(input_paths) <= 1 or batch_size <= 0 or len(input_paths) <= batch_size:
        return call_mineru(input_paths, config)

    markdown_parts: list[str] = []
    total_batches = (len(input_paths) + batch_size - 1) // batch_size

    for batch_index, start in enumerate(range(0, len(input_paths), batch_size), start=1):
        batch = input_paths[start : start + batch_size]
        print(f"[MinerU] Batch {batch_index}/{total_batches}: {len(batch)} image(s)")
        batch_markdown = call_mineru(
            batch,
            config,
            output_name=f"markdown-output-batch-{batch_index:03d}.md",
        )
        markdown_parts.append(batch_markdown)

    markdown = "\n\n".join(markdown_parts).strip()
    out_path = config.output_dir / "markdown-output.md"
    out_path.write_text(markdown, encoding="utf-8")
    print(f"[MinerU] Combined markdown saved to {out_path} ({len(markdown)} chars)")
    return markdown


## Semantic Chunking

Python improvement: heading detection is accent-insensitive and table chunks are split on
markdown-friendly boundaries instead of arbitrary character windows.

In [ ]:
def repair_mojibake(text: str) -> str:
    text = text or ""
    candidates = [text]
    for encoding in ("latin1", "cp1252"):
        try:
            candidates.append(text.encode(encoding).decode("utf-8"))
        except Exception:
            pass

    bad_markers = ["\u00c3", "\u00c2", "\u00c4", "\u00c1", "\ufffd"]
    def score(candidate: str) -> int:
        bad = sum(candidate.count(marker) for marker in bad_markers)
        return sum(ord(ch) > 127 for ch in candidate) - bad * 20

    return max(candidates, key=score)


def remove_diacritics(text: str) -> str:
    text = repair_mojibake(text or "")
    normalized = unicodedata.normalize("NFD", text)
    without_marks = "".join(char for char in normalized if unicodedata.category(char) != "Mn")
    return without_marks.replace("\u0111", "d").replace("\u0110", "D").lower()


@dataclass(frozen=True)
class TablePattern:
    key: str
    pattern: re.Pattern[str]
    next_keys: tuple[str, ...]


@dataclass(frozen=True)
class StandardPatterns:
    audit_pattern: re.Pattern[str]
    cutoff_pattern: re.Pattern[str]
    tables: tuple[TablePattern, ...]


def heading_pattern(text: str) -> re.Pattern[str]:
    # MinerU may output headings as markdown headers, plain text, uppercase text, or OCR-noisy lines.
    return re.compile(rf"(?:^|\n)\s*(?:#+\s*)?(?:[A-ZIVX\d]+[.)]?\s*)?{text}", re.I)


STANDARD_PATTERNS: dict[str, StandardPatterns] = {
    "VAS": StandardPatterns(
        audit_pattern=heading_pattern(r"bao cao kiem toan doc lap(?!\s*\(?tiep theo\)?)"),
        cutoff_pattern=heading_pattern(r"thuyet minh bao cao tai chinh"),
        tables=(
            TablePattern(
                key="BCDKT",
                pattern=heading_pattern(r"(?:bang can doi ke toan|bao cao tinh hinh tai chinh)(?!\s*\(?tiep theo\)?)"),
                next_keys=("BCHDKD", "BCLCTT", "_cutoff"),
            ),
            TablePattern(
                key="BCHDKD",
                pattern=heading_pattern(r"bao cao ket qua hoat dong(?: kinh doanh)?(?!\s*\(?tiep theo\)?)"),
                next_keys=("BCLCTT", "_cutoff"),
            ),
            TablePattern(
                key="BCLCTT",
                pattern=heading_pattern(r"bao cao luu chuyen tien te(?!\s*\(?tiep theo\)?)"),
                next_keys=("_cutoff",),
            ),
        ),
    ),
    "IFRS": StandardPatterns(
        audit_pattern=heading_pattern(r"independent auditors'? report(?!\s*\(?continued\)?)"),
        cutoff_pattern=heading_pattern(r"notes to (the )?(consolidated )?financial statements"),
        tables=(
            TablePattern(
                key="CSFP",
                pattern=heading_pattern(r"(consolidated\s+)?statement of financial position(?!\s*\(?continued\)?)"),
                next_keys=("CSPLCI", "CSCE", "CSCF", "_cutoff"),
            ),
            TablePattern(
                key="CSPLCI",
                pattern=heading_pattern(r"(consolidated\s+)?statement of profit or loss( and other comprehensive income)?(?!\s*\(?continued\)?)"),
                next_keys=("CSCE", "CSCF", "_cutoff"),
            ),
            TablePattern(
                key="CSCE",
                pattern=heading_pattern(r"(consolidated\s+)?statement of changes in equity(?!\s*\(?continued\)?)"),
                next_keys=("CSCF", "_cutoff"),
            ),
            TablePattern(
                key="CSCF",
                pattern=heading_pattern(r"(consolidated\s+)?statement of cash flows(?!\s*\(?continued\)?)"),
                next_keys=("_cutoff",),
            ),
        ),
    ),
}


TABLE_CONTEXT_TERMS = [
    "chi tieu",
    "ma so",
    "thuyet minh",
    "so cuoi nam",
    "so dau nam",
    "nam nay",
    "nam truoc",
    "current year",
    "previous year",
    "code",
    "note",
]


def find_last_match(pattern: re.Pattern[str], text: str) -> re.Match[str] | None:
    last: re.Match[str] | None = None
    for match in pattern.finditer(text):
        last = match
    return last


def rewind_to_line_start(text: str, idx: int) -> int:
    while idx > 0 and text[idx - 1] != "\n":
        idx -= 1
    return idx


def match_content_index(text: str, idx: int) -> int:
    # Patterns often include (^|\n); if the regex starts on a newline, move to the content line.
    if idx < len(text) and text[idx] == "\n":
        return idx + 1
    return idx


def has_table_context(search_text: str, idx: int, window: int = 2500) -> bool:
    context = search_text[idx : idx + window]
    return any(term in context for term in TABLE_CONTEXT_TERMS)


def find_statement_position(pattern: re.Pattern[str], normalized: str, search_text: str, search_from: int) -> int | None:
    matches = list(pattern.finditer(search_text[search_from:]))
    if not matches:
        return None

    # Prefer the first heading that is followed by table-like columns, avoiding TOC rows when fallback uses early pages.
    for match in matches:
        idx = match_content_index(search_text, match.start() + search_from)
        if has_table_context(search_text, idx):
            return rewind_to_line_start(normalized, idx)

    idx = match_content_index(search_text, matches[0].start() + search_from)
    return rewind_to_line_start(normalized, idx)



def markdown_blocks_by_file_label(normalized: str, standard: str) -> dict[str, str]:
    """Map MinerU `<!-- filename -->` blocks to statement keys from subset-PDF filenames."""
    pattern = re.compile(r"<!--\s*([^>]+?)\s*-->\s*")
    matches = list(pattern.finditer(normalized))
    if not matches:
        return {}

    blocks: dict[str, str] = {}
    known_keys = set(toc_table_patterns(standard).keys())
    for idx, match in enumerate(matches):
        label = match.group(1)
        start = match.end()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(normalized)
        label_norm = remove_diacritics(label)
        block = normalized[match.start():end].strip()

        if "meta_pages" in label_norm or re.search(r"(?:^|[_\-\s])meta(?:[_\-\s]|$)", label_norm):
            blocks["META"] = block
            continue

        for table_key in known_keys:
            if table_key.lower() in label_norm:
                blocks[table_key] = block
                break
        else:
            inferred = toc_entry_table_key(label, standard)
            if inferred:
                blocks[inferred] = block

    if blocks:
        print("[Chunk] File-label chunks:", {key: len(value) for key, value in blocks.items()})
    return blocks


ANCHOR_PATTERNS: dict[str, list[re.Pattern[str]]] = {
    "BCDKT": [
        heading_pattern(r"tai.{0,3}san.{0,8}ngan.{0,3}han"),
        heading_pattern(r"tai.{0,3}san.{0,8}dai.{0,3}han"),
        heading_pattern(r"no.{0,3}phai.{0,3}tra"),
        heading_pattern(r"von.{0,3}chu.{0,3}so.{0,3}huu"),
        heading_pattern(r"tong.{0,6}cong.{0,6}tai.{0,3}san"),
        heading_pattern(r"tong.{0,6}cong.{0,6}nguon.{0,3}von"),
    ],
    "BCHDKD": [
        heading_pattern(r"doanh.{0,3}thu.{0,8}thuan"),
        heading_pattern(r"gia.{0,3}von.{0,8}hang.{0,3}ban"),
        heading_pattern(r"loi.{0,3}nhuan.{0,8}gop"),
        heading_pattern(r"loi.{0,3}nhuan.{0,8}thuan.{0,20}hoat.{0,3}dong.{0,8}kinh.{0,3}doanh"),
        heading_pattern(r"tong.{0,8}loi.{0,3}nhuan.{0,20}truoc.{0,3}thue"),
        heading_pattern(r"thu.{0,3}nhap.{0,3}lai.{0,12}cac.{0,3}khoan.{0,3}thu.{0,3}nhap.{0,3}tuong.{0,3}tu"),
        heading_pattern(r"chi.{0,3}phi.{0,3}lai.{0,12}cac.{0,3}chi.{0,3}phi.{0,3}tuong.{0,3}tu"),
        heading_pattern(r"thu.{0,3}nhap.{0,3}lai.{0,3}thuan"),
        heading_pattern(r"lai.{0,3}lo.{0,3}thuan.{0,10}hoat.{0,3}dong.{0,3}dich.{0,3}vu"),
        heading_pattern(r"chi.{0,3}phi.{0,3}du.{0,3}phong.{0,3}rui.{0,3}ro.{0,3}tin.{0,3}dung"),
        heading_pattern(r"loi.{0,3}nhuan.{0,3}sau.{0,3}thue"),
    ],
    "BCLCTT": [
        heading_pattern(r"luu.{0,3}chuyen.{0,3}tien.{0,10}hoat.{0,3}dong.{0,8}kinh.{0,3}doanh"),
        heading_pattern(r"luu.{0,3}chuyen.{0,3}tien.{0,10}dau.{0,3}tu"),
        heading_pattern(r"luu.{0,3}chuyen.{0,3}tien.{0,10}tai.{0,3}chinh"),
        heading_pattern(r"tien.{0,8}tuong.{0,8}duong.{0,8}tien.{0,8}cuoi.{0,3}ky"),
        heading_pattern(r"luu.{0,3}chuyen.{0,3}tien.{0,3}te"),
    ],
    "CSFP": [heading_pattern(r"assets"), heading_pattern(r"liabilities"), heading_pattern(r"equity")],
    "CSPLCI": [heading_pattern(r"revenue"), heading_pattern(r"profit before tax"), heading_pattern(r"profit for the year")],
    "CSCE": [heading_pattern(r"balance at"), heading_pattern(r"share capital"), heading_pattern(r"retained earnings")],
    "CSCF": [heading_pattern(r"cash flows from operating activities"), heading_pattern(r"cash and cash equivalents")],
}


def find_anchor_position(table_key: str, normalized: str, search_text: str, search_from: int) -> int | None:
    candidates: list[int] = []
    for pattern in ANCHOR_PATTERNS.get(table_key, []):
        match = pattern.search(search_text[search_from:])
        if match:
            candidates.append(match_content_index(search_text, match.start() + search_from))
    if not candidates:
        return None

    idx = min(candidates)
    # Anchor fallback starts at the anchor line. Rewinding to a previous generic heading can
    # accidentally pull the preceding statement into this chunk when OCR mangles the title.
    return rewind_to_line_start(normalized, idx)


def semantic_chunk(markdown: str, standard: str = CONFIG.standard) -> dict[str, Any]:
    cfg = STANDARD_PATTERNS.get(standard.upper(), STANDARD_PATTERNS["VAS"])
    normalized = re.sub(r"\n{3,}", "\n\n", markdown.replace("\r\n", "\n")).strip()
    search_text = remove_diacritics(normalized)

    file_label_chunks = markdown_blocks_by_file_label(normalized, standard)

    audit_match = find_last_match(cfg.audit_pattern, search_text)
    if audit_match:
        search_from = audit_match.end()
        print(f"[Chunk] Auditor report found. Searching after index {search_from}")
    else:
        search_from = 0
        print("[Chunk] Auditor report not found. Searching from document start")

    positions: dict[str, int | None] = {table.key: None for table in cfg.tables}
    positions["_cutoff"] = find_statement_position(cfg.cutoff_pattern, normalized, search_text, search_from)

    for table in cfg.tables:
        pos = find_statement_position(table.pattern, normalized, search_text, search_from)
        if pos is None:
            pos = find_anchor_position(table.key, normalized, search_text, search_from)
            if pos is not None:
                print(f"[Chunk] {table.key}: anchor fallback at {pos}")
        positions[table.key] = pos
        print(f"[Chunk] {table.key}: {positions[table.key] if positions[table.key] is not None else 'not found'}")

    print(f"[Chunk] _cutoff: {positions['_cutoff'] if positions['_cutoff'] is not None else 'not found'}")

    def slice_chunk(from_key: str, next_keys: tuple[str, ...]) -> str | None:
        from_idx = positions[from_key]
        if from_idx is None:
            return None

        to_idx = len(normalized)
        candidates = [positions[nk] for nk in next_keys if positions.get(nk) is not None and positions[nk] > from_idx]
        if candidates:
            to_idx = min(candidates)
        return normalized[from_idx:to_idx].strip()

    table_chunks = {table.key: slice_chunk(table.key, table.next_keys) for table in cfg.tables}

    # Prefer file-label chunks from per-statement subset PDFs. They survive title variations and OCR errors.
    for table_key, block in file_label_chunks.items():
        if table_key in table_chunks:
            table_chunks[table_key] = block

    # Ordered fallback: if BCLCTT is missing after BCHDKD, use the tail after BCHDKD only when cash-flow anchors appear.
    if standard.upper() == "VAS" and not table_chunks.get("BCLCTT") and positions.get("BCHDKD") is not None:
        tail_start = positions["BCHDKD"]
        tail = normalized[tail_start:]
        tail_search = remove_diacritics(tail)
        anchor = find_anchor_position("BCLCTT", tail, tail_search, 0)
        if anchor is not None:
            table_chunks["BCLCTT"] = tail[anchor:].strip()
            print("[Chunk] BCLCTT recovered from cash-flow anchor in BCHDKD tail")

    found_positions = sorted(pos for key, pos in positions.items() if key != "_cutoff" and pos is not None)
    meta_chunk = file_label_chunks.get("META")
    if meta_chunk:
        print(f"[Chunk] META: file-label chunk {len(meta_chunk)} chars")
    else:
        meta_chunk = normalized[: found_positions[0] if found_positions else min(2000, len(normalized))].strip()
    any_table_found = any(table_chunks.values())

    print("[Chunk] Sizes:", {key: len(value or "") for key, value in table_chunks.items()})
    return {
        "normalized": normalized,
        "meta_chunk": meta_chunk,
        "table_chunks": table_chunks,
        "any_table_found": any_table_found,
    }


def split_text_smart(text: str, max_chars: int = CONFIG.chunk_max_chars) -> list[str]:
    if not text:
        return []
    if len(text) <= max_chars:
        return [text]

    # Split on high-level boundaries, not every table row. This keeps headers and nearby rows together.
    blocks = re.split(r"(?=\n#{1,6}\s|\n<!--)", text)
    pieces: list[str] = []
    current = ""

    def flush_current() -> None:
        nonlocal current
        if current.strip():
            pieces.append(current.strip())
        current = ""

    for block in blocks:
        if not block:
            continue
        if len(current) + len(block) <= max_chars:
            current += block
            continue
        flush_current()

        if len(block) <= max_chars:
            current = block
            continue

        # Large block fallback: split by line, preserving markdown table rows as much as possible.
        line_chunk = ""
        for line in block.splitlines(keepends=True):
            if len(line_chunk) + len(line) > max_chars and line_chunk.strip():
                pieces.append(line_chunk.strip())
                line_chunk = ""
            line_chunk += line
        if line_chunk.strip():
            pieces.append(line_chunk.strip())

    flush_current()
    return pieces


## Extraction Prompts

In [ ]:
STRICT_SYSTEM_VI = (
    "Bạn là chuyên gia trích xuất dữ liệu báo cáo tài chính. "
    "Chỉ trả về raw JSON hợp lệ, tuyệt đối không có markdown fence, không có giải thích thêm."
)

STRICT_SYSTEM_EN = (
    "You are a financial statement data extraction expert. "
    "Return only valid raw JSON. Do not include markdown fences or explanations."
)

ROW_SCHEMA = {
    "REPORT_INDEX": "",
    "REPORT_NAME": "",
    "REPORT_CODE": "",
    "REPORT_NOTE": "",
    "REPORT_BALANCE_OPEN": "",
    "REPORT_BALANCE_CLOSE": "",
}

FIELDS_SCHEMA_VAS = {
    "REPORT_INDEX": "Số TT",
    "REPORT_NAME": "Chỉ tiêu",
    "REPORT_CODE": "Mã số",
    "REPORT_NOTE": "Thuyết minh",
    "REPORT_BALANCE_OPEN": "",
    "REPORT_BALANCE_CLOSE": "",
}

FIELDS_SCHEMA_IFRS = {
    "REPORT_INDEX": "Index",
    "REPORT_NAME": "Name",
    "REPORT_CODE": "Code",
    "REPORT_NOTE": "Note",
    "REPORT_BALANCE_OPEN": "",
    "REPORT_BALANCE_CLOSE": "",
}

VAS_RULES = """
Quy t?c:
- Ch? tr?ch xu?t b?ng th?c s? c? trong ?o?n markdown.
- REPORT_INDEX kh?ng c? d?u ch?m (I kh?ng ph?i I.). N?u kh?ng c? th? d?ng "".
- REPORT_CODE l? gi? tr? c?t m? s? ri?ng, kh?ng nh?m v?i REPORT_INDEX.
- N?u b?ng kh?ng c? c?t "M? s?" ri?ng (v? d? b?o c?o ng?n h?ng/TCTD ch? c? STT, Ch? ti?u, Thuy?t minh) th? REPORT_CODE ph?i l? "" cho m?i d?ng.
- REPORT_INDEX l?y t? c?t STT/S? TT n?u c?.
- V?i b?o c?o ng?n h?ng/TCTD, "B?O C?O K?T QU? HO?T ??NG H?P NH?T" v?n l? BCHDKD d? ti?u ?? kh?ng c? ch? "KINH DOANH".
- NAME ph?i l?y ch?nh x?c d?ng ti?u ?? l?n trong markdown; kh?ng t? th?m "GI?A NI?N ??", "KINH DOANH" ho?c t? kh?c n?u ti?u ?? kh?ng c?.
- ? tr?ng tr? v? "" v? kh?ng d?ng null.
- S? ?m d?ng (10.043.543) ph?i chuy?n th?nh "-10.043.543".
- OPEN l? th?i ?i?m tr??c, CLOSE l? th?i ?i?m sau. L?y hai n?m g?n nh?t v? tr?n n?m.
"""


def meta_prompt(meta_chunk: str, standard: str) -> tuple[str, str]:
    if standard.upper() == "IFRS":
        return (
            STRICT_SYSTEM_EN,
            f"""Extract company metadata from this markdown:

{meta_chunk}

Return exactly this JSON structure:
{{"COMPANY_NAME":"","COMPANY_ADDRESS":"","DATE":""}}""",
        )

    return (
        STRICT_SYSTEM_VI,
        f"""Từ đoạn markdown dưới đây hãy trích xuất thông tin công ty:

{meta_chunk}

Trả về đúng cấu trúc JSON sau:
{{"COMPANY_NAME":"","COMPANY_ADDRESS":"","DATE":""}}""",
    )


def table_prompt(
    table_key: str,
    markdown_piece: str,
    piece_index: int,
    total_pieces: int,
    standard: str,
) -> tuple[str, str]:
    is_first = piece_index == 0
    part = f"({piece_index + 1}/{total_pieces})"
    standard = standard.upper()

    if standard == "IFRS":
        fields_schema = json.dumps(FIELDS_SCHEMA_IFRS, ensure_ascii=False)
        row_schema = json.dumps(ROW_SCHEMA, ensure_ascii=False)
        if is_first:
            prompt = f"""Extract {table_key} from this markdown part {part}:

{markdown_piece}

Return this JSON structure:
{{"{table_key}":{{"NAME":"<full statement name from document>","FIELDS":{fields_schema},"ROWS":[{row_schema}]}}}}

{IFRS_RULES}"""
        else:
            prompt = f"""Continue extracting ROWS from markdown part {part} for table {table_key}. Do not repeat rows already extracted.

{markdown_piece}

Return only a JSON array:
[{row_schema}]

{IFRS_RULES}"""
        return STRICT_SYSTEM_EN, prompt

    fields_schema = json.dumps(FIELDS_SCHEMA_VAS, ensure_ascii=False)
    row_schema = json.dumps(ROW_SCHEMA, ensure_ascii=False)
    if is_first:
        prompt = f"""Trích xuất {table_key} từ markdown sau {part}:

{markdown_piece}

Cấu trúc JSON:
{{"{table_key}":{{"NAME":"<tên đầy đủ từ tài liệu>","FIELDS":{fields_schema},"ROWS":[{row_schema}]}}}}

{VAS_RULES}"""
    else:
        prompt = f"""Tiếp tục trích xuất ROWS từ đoạn markdown tiếp theo {part} của bảng {table_key}. Không lặp lại dòng đã trích xuất.

{markdown_piece}

Chỉ trả về mảng JSON:
[{row_schema}]

{VAS_RULES}"""
    return STRICT_SYSTEM_VI, prompt

## Model Extraction

In [ ]:
def extract_metadata(meta_chunk: str, config: Config = CONFIG) -> dict[str, Any]:
    system, prompt = meta_prompt(meta_chunk, config.standard)
    print(f"[Extract] META ({len(prompt)} chars)")
    raw = ollama_generate(
        model=config.extraction_model,
        system=system,
        prompt=prompt,
        options={"temperature": 0.1, "num_predict": 2048, "num_ctx": 8192},
        config=config,
    )
    parsed = parse_json_response("META", raw)
    return parsed if isinstance(parsed, dict) else {}


def source_has_report_code_column(source_markdown: str, standard: str = CONFIG.standard) -> bool:
    normalized = remove_diacritics(source_markdown or "")
    if standard.upper() == "IFRS":
        return bool(re.search(r"\bcode\b", normalized))
    # Distinguish Vietnamese "Ma so" from "Mau so" form numbers such as B02a/TCTD.
    return bool(re.search(r"\bma\s*so\b", normalized))


def clean_report_index(value: Any) -> str:
    text = str(value or "").strip()
    return re.sub(r"\.+$", "", text).strip()


def looks_like_stt(value: Any) -> bool:
    text = clean_report_index(value)
    return bool(re.fullmatch(r"[A-Z]|[IVXLCDM]+|\d{1,2}|[a-z]", text, flags=re.I))


def detect_statement_name(table_key: str, source_markdown: str, standard: str = CONFIG.standard) -> str | None:
    if not source_markdown:
        return None

    if standard.upper() == "IFRS":
        patterns = {
            "CSFP": [r"statement of financial position"],
            "CSPLCI": [r"statement of profit or loss", r"comprehensive income"],
            "CSCE": [r"statement of changes in equity"],
            "CSCF": [r"statement of cash flows"],
        }
    else:
        patterns = {
            "BCDKT": [r"bao cao tinh hinh tai chinh", r"bang can doi ke toan"],
            "BCHDKD": [r"bao cao ket qua hoat dong(?: kinh doanh)?"],
            "BCLCTT": [r"bao cao luu chuyen tien te"],
        }

    wanted = [re.compile(pattern, re.I) for pattern in patterns.get(table_key, [])]
    if not wanted:
        return None

    for raw_line in source_markdown.splitlines():
        line = re.sub(r"<!--.*?-->", "", raw_line).strip()
        line = re.sub(r"^#+\s*", "", line).strip()
        line = line.strip("| ").strip()
        if not line:
            continue
        normalized_line = remove_diacritics(line)
        if "tiep theo" in normalized_line or "continued" in normalized_line:
            continue
        if any(pattern.search(normalized_line) for pattern in wanted):
            return re.sub(r"\s+", " ", line).strip()

    return None




def infer_report_date_from_markdown(source_markdown: str) -> str | None:
    """Prefer the reporting date near statement headings over signature/approval dates."""
    for raw_line in (source_markdown or "").splitlines():
        line = re.sub(r"^#+\s*", "", raw_line).strip("| ").strip()
        if not line:
            continue
        normalized_line = remove_diacritics(line)
        if "tai ngay" not in normalized_line and "as at" not in normalized_line:
            continue

        slash_date = re.search(r"\b\d{1,2}[/.-]\d{1,2}[/.-]\d{4}\b", line)
        if slash_date:
            return slash_date.group(0)

        words_date = re.search(r"\b\d{1,2}\s+\S+\s+\d{1,2}\s+\S+\s+\d{4}\b", line, flags=re.I)
        if words_date:
            return words_date.group(0)

    return None

def postprocess_extracted_table(
    table_obj: dict[str, Any],
    table_key: str,
    source_markdown: str,
    standard: str = CONFIG.standard,
) -> dict[str, Any]:
    table = table_obj.get(table_key)
    if not isinstance(table, dict):
        return table_obj

    detected_name = detect_statement_name(table_key, source_markdown, standard)
    if detected_name:
        table["NAME"] = detected_name

    rows = table.get("ROWS")
    if isinstance(rows, list):
        has_code_column = source_has_report_code_column(source_markdown, standard)
        for row in rows:
            if not isinstance(row, dict):
                continue
            row["REPORT_INDEX"] = clean_report_index(row.get("REPORT_INDEX"))
            if not has_code_column:
                code_value = row.get("REPORT_CODE", "")
                if not row.get("REPORT_INDEX") and looks_like_stt(code_value):
                    row["REPORT_INDEX"] = clean_report_index(code_value)
                row["REPORT_CODE"] = ""

    return table_obj


def extract_table(table_key: str, source_markdown: str, config: Config = CONFIG) -> dict[str, Any] | None:
    pieces = split_text_smart(source_markdown, config.chunk_max_chars)
    if not pieces:
        return None

    print(f"[Extract] {table_key}: {len(pieces)} piece(s), {len(source_markdown)} chars")
    first_obj: dict[str, Any] | None = None
    all_rows: list[dict[str, Any]] = []

    for idx, piece in enumerate(pieces):
        system, prompt = table_prompt(table_key, piece, idx, len(pieces), config.standard)
        print(f"[Extract] {table_key}[{idx + 1}/{len(pieces)}] ({len(prompt)} chars)")
        raw = ollama_generate(
            model=config.extraction_model,
            system=system,
            prompt=prompt,
            options={"temperature": 0.1, "num_predict": 8192, "num_ctx": 8192},
            config=config,
        )
        parsed = parse_json_response(f"{table_key}[{idx + 1}/{len(pieces)}]", raw)
        if parsed is None:
            continue

        if idx == 0 and isinstance(parsed, dict):
            first_obj = parsed
            rows = parsed.get(table_key, {}).get("ROWS", [])
            if isinstance(rows, list):
                all_rows.extend(rows)
        elif isinstance(parsed, list):
            all_rows.extend(row for row in parsed if isinstance(row, dict))
        elif isinstance(parsed, dict):
            rows = parsed.get(table_key, {}).get("ROWS", [])
            if isinstance(rows, list):
                all_rows.extend(rows)

    if not first_obj or table_key not in first_obj:
        return None

    first_obj[table_key]["ROWS"] = all_rows
    cleaned = empty_none_values(first_obj)
    return postprocess_extracted_table(cleaned, table_key, source_markdown, config.standard)


## Run Pipeline

Update `CONFIG.pdf_path`, then run `result = run_pipeline(CONFIG)`.

In [ ]:
def run_pipeline(config: Config = CONFIG, *, pdf_path: str | Path | None = None) -> dict[str, Any]:
    if pdf_path is not None:
        config = replace(config, pdf_path=Path(pdf_path))

    config = replace(config, mineru_url=normalize_mineru_url(config.mineru_url))
    config.output_dir.mkdir(parents=True, exist_ok=True)

    pdf_path = Path(config.pdf_path)
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    print("[Pipeline] Rendering pages for TOC scan...")
    toc_images = render_pdf_pages_to_images(
        pdf_path,
        config.toc_scan_pages,
        output_dir=config.output_dir,
        dpi=config.render_dpi,
    )

    table_of_contents = detect_table_of_contents(toc_images, config)
    unload_ollama_model(config.vision_model, config)

    total_pages = get_pdf_page_count(pdf_path)
    selected_page_numbers, selected_ranges = select_statement_pages_from_toc(
        table_of_contents,
        total_pages=total_pages,
        config=config,
    )

    metadata_page_numbers, metadata_ranges = select_metadata_pages_from_toc(
        table_of_contents,
        total_pages=total_pages,
        config=config,
    )
    metadata_file = create_metadata_subset_pdf(
        pdf_path,
        metadata_page_numbers,
        config.output_dir,
    ) if selected_ranges else None

    statement_files = create_statement_subset_pdfs(
        pdf_path,
        selected_ranges,
        selected_page_numbers,
        config.output_dir,
    )
    mineru_pdf_paths = ([Path(metadata_file["pdf_path"])] if metadata_file else []) + [
        Path(item["pdf_path"]) for item in statement_files
    ]

    selection_manifest_path = config.output_dir / "selected-pages.json"
    save_json(
        {
            "source_pdf": str(pdf_path),
            "total_pages": total_pages,
            "metadata_pages": metadata_page_numbers,
            "metadata_ranges": metadata_ranges,
            "metadata_file": metadata_file,
            "selected_pages": selected_page_numbers,
            "selected_ranges": selected_ranges,
            "statement_files": statement_files,
            "mineru_files": [str(path) for path in mineru_pdf_paths],
            "table_of_contents": table_of_contents,
        },
        selection_manifest_path,
    )
    print(f"[Pipeline] Selected page manifest saved to {selection_manifest_path}")

    if config.mineru_input_mode.lower() == "images":
        mineru_image_page_numbers = sorted(set(metadata_page_numbers + selected_page_numbers))
        print(f"[Pipeline] Rendering {len(mineru_image_page_numbers)} metadata + TOC-selected page image(s) for MinerU...")
        selected_images = render_pdf_page_numbers_to_images(
            pdf_path,
            mineru_image_page_numbers,
            output_dir=config.output_dir,
            dpi=config.render_dpi,
        )
        print("[Pipeline] Sending metadata + TOC-selected rendered images to MinerU in batches...")
        try:
            markdown = call_mineru_batched([page.path for page in selected_images], config)
        except requests.exceptions.Timeout as exc:
            if not config.mineru_pdf_fallback:
                raise
            print(f"[Pipeline] MinerU image mode timed out: {exc}")
            print("[Pipeline] Falling back to per-statement subset PDFs for MinerU...")
            markdown = call_mineru(mineru_pdf_paths, config)
    else:
        print("[Pipeline] Sending metadata + per-statement TOC-selected subset PDFs to MinerU...")
        markdown = call_mineru(mineru_pdf_paths, config)

    print("[Pipeline] Chunking markdown...")
    chunks = semantic_chunk(markdown, config.standard)

    print("[Pipeline] Extracting metadata...")
    meta = extract_metadata(chunks["meta_chunk"], config)
    final_result: dict[str, Any] = {}

    for key in ("COMPANY_NAME", "COMPANY_ADDRESS", "DATE"):
        if meta.get(key):
            final_result[key] = meta[key]

    cfg = STANDARD_PATTERNS.get(config.standard.upper(), STANDARD_PATTERNS["VAS"])
    for table in cfg.tables:
        source = chunks["table_chunks"].get(table.key)
        if not source and not chunks["any_table_found"]:
            source = chunks["normalized"]

        if not source:
            print(f"[Pipeline] Skipping {table.key}; chunk not found")
            continue

        table_obj = extract_table(table.key, source, config)
        if table_obj and table_obj.get(table.key):
            final_result[table.key] = table_obj[table.key]

    report_date = infer_report_date_from_markdown(
        chunks["table_chunks"].get("BCDKT")
        or chunks["table_chunks"].get("CSFP")
        or ""
    )
    if report_date:
        final_result["DATE"] = report_date

    unload_ollama_model(config.extraction_model, config)

    result_path = config.output_dir / "extraction-result.json"
    save_json(final_result, result_path)
    print(f"[Pipeline] Result saved to {result_path}")
    print("[Pipeline] Final keys:", list(final_result))
    return final_result


In [ ]:
# Example usage:

CONFIG = replace(CONFIG, pdf_path=Path(r"/content/VAS-09.pdf"), standard="VAS")
result = run_pipeline(CONFIG)
result

[Pipeline] Rendering pages for TOC scan...
[Render] Rendered 3/52 page(s) for TOC scan to /content/outputs/VAS-09_toc_scan_pages
[TOC] Scanning page 1
[TOC] Scanning page 2
[TOC] Scanning page 3
[TOC] No table of contents found
[Ollama] Cloud host in use; no local unload needed for qwen3-vl:235b
[TOC] No TOC. Falling back to first 30 page(s): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
[TOC] Selected metadata pages for MinerU: [1, 2, 3]
[Render] Saved TOC-selected subset PDF with pages [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30] to /content/outputs/VAS-09_statement_pdfs/VAS-09_FALLBACK_pages_001_030.pdf
[Render] Statement subset PDFs: [{'table_key': 'FALLBACK', 'title': 'fallback selected pages', 'start_page': 1, 'end_page': 30, 'pages': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30], 'pd

{'COMPANY_NAME': 'Ngân hàng Thương mại Cổ phần Tiên Phong',
 'COMPANY_ADDRESS': 'Tòa nhà TPBank, Số 57 Lý Thường Kiệt, Phường Cửa Nam, Thành phố Hà Nội, Việt Nam',
 'DATE': '30 tháng 06 năm 2025',
 'BCDKT': {'NAME': 'BÁO CÁO TÌNH HÌNH TÀI CHÍNH HỢP NHẤT',
  'FIELDS': {'REPORT_INDEX': 'Số TT',
   'REPORT_NAME': 'Chỉ tiêu',
   'REPORT_CODE': 'Mã số',
   'REPORT_NOTE': 'Thuyết minh',
   'REPORT_BALANCE_OPEN': 'REPORT_BALANCE_OPEN',
   'REPORT_BALANCE_CLOSE': 'REPORT_BALANCE_CLOSE'},
  'ROWS': [{'REPORT_INDEX': 'I',
    'REPORT_NAME': 'Tiền mặt vàng bạc đá quý',
    'REPORT_CODE': '',
    'REPORT_NOTE': ' ',
    'REPORT_BALANCE_OPEN': '1.551.561',
    'REPORT_BALANCE_CLOSE': '1.292.735'},
   {'REPORT_INDEX': 'II',
    'REPORT_NAME': 'Tiền gửi tại NHNN',
    'REPORT_CODE': '',
    'REPORT_NOTE': ' ',
    'REPORT_BALANCE_OPEN': '10.000.595',
    'REPORT_BALANCE_CLOSE': '22.708.369'},
   {'REPORT_INDEX': 'III',
    'REPORT_NAME': 'Tiền vàng gửi tại các TCTD khác và cho vay các TCTD khác',
   